# PT Vorarbeiten
Run once (or when historical data changes). Produces pre-computed parquet files and notepad flags used by the fast HTML notebook.

## 1. Imports

In [1]:
# Standard library
import sys, time, ast, json, re, warnings
from datetime import datetime, timedelta
from itertools import cycle
from io import StringIO

# Data
import numpy as np
import pandas as pd
import pytz
from tqdm import tqdm
import json

# Stats / ML
from scipy.stats import ttest_ind
from statsmodels.nonparametric.smoothers_lowess import lowess
from textblob import TextBlob

# Scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import (
    OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, GroupShuffleSplit, GroupKFold, GridSearchCV
)
from sklearn.metrics import mean_squared_error, roc_auc_score, make_scorer
from sklearn.impute import KNNImputer
from sklearn.base import clone

# Plotting
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import cm
from matplotlib.lines import Line2D
import os

from google.colab import userdata
import json
import math

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import random

import re
import unicodedata
import pandas as pd
import requests
from datetime import datetime
from google.colab import userdata

In [2]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 10.1 MB/s eta 0:00:00


In [3]:
import anthropic

In [ ]:
# papermill parameters — injected by PT_WORKFLOW.ipynb when run automatically
ANTHROPIC_API_KEY = None


## 2. Mount Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Make ANTHROPIC_API_KEY available to sub-functions (e.g. compute_notepad_flags)
import os
if ANTHROPIC_API_KEY:
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY


## 3. Load Data

In [5]:
# ── Historical data ──────────────────────────────────────────────────────────
BASE = "/content/drive/MyDrive/PT" if os.path.exists("/content/drive/MyDrive/PT") else "."

runners = (pd.read_parquet(f"{BASE}/runners.parquet")
             .drop_duplicates(subset=['horseName', 'raceId']))

webTips = (pd.read_parquet(f"{BASE}/webTips.parquet")
             .drop_duplicates(subset=['meetingId', 'raceId']))
webTips = webTips[webTips['text'].notna()]

odds  = pd.read_parquet(f"{BASE}/odds.parquet").drop_duplicates()
races = pd.read_parquet(f"{BASE}/races.parquet").drop_duplicates()

In [6]:
# ── Today's data ─────────────────────────────────────────────────────────────

runners_tdy  = (pd.read_parquet(f"{BASE}/runners_tdy.parquet")
                  .drop_duplicates(subset=['horseName', 'raceId']))
webTips_tdy  = (pd.read_parquet(f"{BASE}/webTips_tdy.parquet")
                  .drop_duplicates(subset=['meetingId', 'raceId']))
odds_tdy     = pd.read_parquet(f"{BASE}/odds_tdy.parquet").drop_duplicates()

races_tdy    = pd.read_parquet(f"{BASE}/races_tdy.parquet").drop_duplicates()

today_tips    = pd.read_parquet(f"{BASE}/today_tips.parquet").drop_duplicates()

In [7]:
import json
from datetime import date

if os.path.exists(f"{BASE}/going_overrides.json"):
    with open(f"{BASE}/going_overrides.json") as f:
        overrides = json.load(f)

    override_date = overrides.get('date', '')
    today_str = str(date.today())

    if override_date == today_str:
        for meeting, going in overrides.get('by_meeting', {}).items():
            races_tdy.loc[races_tdy['name_meeting'] == meeting, 'going'] = going

        for race_name, going in overrides.get('by_race_name', {}).items():
            races_tdy.loc[races_tdy['name_race'] == race_name, 'going'] = going

        print(f"✅ Going overrides applied for {today_str}")
    else:
        print(f"⚠️ going_overrides.json is dated '{override_date}', today is '{today_str}' — skipping overrides")
else:
    print("⚠️ No going_overrides.json found — no overrides applied")

✅ Going overrides applied for 2026-04-22


## 4. Merge Dataframes

In [8]:
# Merge historical dataframes
runners = runners.merge(
    races[['id_race', 'class', 'type', 'going', 'date', 'totalPrize',
           'distance', 'name_meeting', 'direction', 'rail', 'surface', 'winnerTime']],
    left_on='raceId', right_on='id_race', how='left'
)
runners = runners.merge(
    odds[['raceId', 'horseName', 'referenceOdd', 'liveOdd']],
    on=['raceId', 'horseName'], how='left'
)
runners = runners.merge(
    webTips[['raceId', 'meetingId', 'text']],
    on=['raceId', 'meetingId'], how='left'
)

# Merge today's runners with race details
runners_tdy = runners_tdy.merge(
    races_tdy[['id_race', 'going', 'distance', 'name_race', 'name_meeting', 'date']],
    left_on='raceId', right_on='id_race', how='left'
)

## 5. Helper Functions

In [9]:
# ── Margin / lengths helpers ─────────────────────────────────────────────────

def build_margin_lookup(runners):
    return {
        (row.raceId, row.ranking): row.margin
        for row in runners.itertuples(index=False)
    }

def margin_to_length_fast(margin, ranking, raceId, margin_lookup):
    if ranking == 1:
        margin_rank2 = margin_lookup.get((raceId, 2), None)
        if margin_rank2 is None:
            return 0
        return -margin_to_length_fast(margin_rank2, 2, raceId, margin_lookup)
    if pd.isnull(margin):
        return 0
    if isinstance(margin, str):
        margin = margin.replace('(', '').replace(')', '')
        margin_map = {
            'DH': 0, 'NEZ': 0.05, 'CTT': 0.1,
            'TETE': 0.2, 'CTE': 0.3, 'ENC': 0.4,
            'LOIN': 99
        }
        if margin in margin_map:
            return margin_map[margin]
        parts = margin.split()
        length = 0.0
        for part in parts:
            if '/' in part:
                num, denom = part.split('/')
                length += float(num) / float(denom)
            else:
                length += float(part)
        return length
    return float(margin)


# ── Distance grouping ─────────────────────────────────────────────────────────

def get_distance_group(m):
    if pd.isna(m):
        return None
    m = int(m)
    if m <= 1000:
        return '0-1000'
    elif m > 3600:
        return '>3600'
    else:
        lower = ((m - 1) // 200) * 200 + 1
        upper = lower + 199
        return f'{lower}-{upper}'


# ── Web tips processing ───────────────────────────────────────────────────────

def rank_horses_by_mention_order(text, horse_names):
    lower_text = text.lower()
    mention_positions = []
    for horse in horse_names:
        pattern = r'\b' + re.escape(horse.lower()) + r'\b'
        match = re.search(pattern, lower_text)
        if match:
            mention_positions.append((match.start(), horse))
    mention_positions.sort()
    return {horse: rank + 1 for rank, (_, horse) in enumerate(mention_positions)}

def process_webtips_simple(webTips, runners):
    all_records = []
    for row in tqdm(webTips.itertuples(), total=len(webTips), desc="Processing tips"):
        race_id = row.raceId
        text = row.text
        horse_names = list(runners[runners['raceId'] == race_id]['horseName'].dropna())
        ranked_horses = rank_horses_by_mention_order(text, horse_names)
        for horse, rank in ranked_horses.items():
            all_records.append({"race_id": race_id, "runner": horse, "webTipRank": rank})
    return pd.DataFrame(all_records)


# ── Prize money ───────────────────────────────────────────────────────────────

def calculate_prize(total_prize, ranking):
    if pd.isna(total_prize) or pd.isna(ranking):
        return 0
    ranking = int(ranking)
    prize_splits = {1: 0.62, 2: 0.22, 3: 0.12, 4: 0.04}
    return total_prize * prize_splits.get(ranking, 0)


# ── Race time parsing ─────────────────────────────────────────────────────────

def parse_race_time(time_str):
    if pd.isna(time_str):
        return None
    time_str = str(time_str).strip()
    pattern_with_minutes = r"(\d+)'(\d+)\"(\d+)"
    pattern_seconds_only = r"'(\d+)\"(\d+)"
    match = re.match(pattern_with_minutes, time_str)
    if match:
        mins, secs, hundredths = match.groups()
        return int(mins) * 60 + int(secs) + int(hundredths) / 10
    match = re.match(pattern_seconds_only, time_str)
    if match:
        secs, hundredths = match.groups()
        return int(secs) + int(hundredths) / 100
    return None

## 6. Rating System

In [10]:
# ── Rating engine ─────────────────────────────────────────────────────────────

kg_per_length = {
    'VERY SLOW': {'0-1000': 1.2, '1001-1200': 1.1, '1201-1400': 1.0,
                  '1401-1600': 0.95, '1601-1800': 0.9, '1801-2000': 0.85,
                  '2001-2200': 0.8, '2201-2400': 0.75, '2401-2600': 0.7,
                  '2601-2800': 0.65, '2801-3000': 0.6, '3001-3200': 0.55,
                  '3201-3400': 0.5, '3401-3600': 0.45, '>3600': 0.4},
    'SLOW':      {'0-1000': 1.3, '1001-1200': 1.2, '1201-1400': 1.1,
                  '1401-1600': 1.05, '1601-1800': 1.0, '1801-2000': 0.95,
                  '2001-2200': 0.9, '2201-2400': 0.85, '2401-2600': 0.8,
                  '2601-2800': 0.75, '2801-3000': 0.7, '3001-3200': 0.65,
                  '3201-3400': 0.6, '3401-3600': 0.55, '>3600': 0.5},
    'FAST':      {'0-1000': 1.5, '1001-1200': 1.4, '1201-1400': 1.3,
                  '1401-1600': 1.2, '1601-1800': 1.15, '1801-2000': 1.1,
                  '2001-2200': 1.05, '2201-2400': 1.0, '2401-2600': 0.95,
                  '2601-2800': 0.9, '2801-3000': 0.85, '3001-3200': 0.8,
                  '3201-3400': 0.75, '3401-3600': 0.7, '>3600': 0.65},
    'VERY FAST': {'0-1000': 1.6, '1001-1200': 1.5, '1201-1400': 1.4,
                  '1401-1600': 1.3, '1601-1800': 1.25, '1801-2000': 1.2,
                  '2001-2200': 1.15, '2201-2400': 1.1, '2401-2600': 1.05,
                  '2601-2800': 1.0, '2801-3000': 0.95, '3001-3200': 0.9,
                  '3201-3400': 0.85, '3401-3600': 0.8, '>3600': 0.75},
    'PSF':       {'0-1000': 1.4, '1001-1200': 1.3, '1201-1400': 1.2,
                  '1401-1600': 1.1, '1601-1800': 1.05, '1801-2000': 1.0,
                  '2001-2200': 0.95, '2201-2400': 0.9, '2401-2600': 0.85,
                  '2601-2800': 0.8, '2801-3000': 0.75, '3001-3200': 0.7,
                  '3201-3400': 0.65, '3401-3600': 0.6, '>3600': 0.55},
}

def lengths_to_rating(l):
    if l < 0:
        raise ValueError("Length must be non-negative")
    if l >= 6.0:
        return 5.0
    x = l / 6.0
    y_norm = np.log1p(9 * x) / np.log(10)
    return 1.0 + 4.0 * y_norm

def signed_bucket(l_signed):
    magnitude = lengths_to_rating(abs(l_signed))
    return magnitude if l_signed >= 0 else -magnitude

def update_ratings_with_history(df, k_factor=1.0, alpha=0.07, initial_ratings=None):
    ratings = dict(initial_ratings) if initial_ratings else {}
    rating_history = []
    df = df.sort_values(by='date')
    df['prize_quantile'] = df.groupby('raceId')['totalPrize_y'].rank(pct=True)
    race_ids = df['raceId'].unique()
    for race_id in tqdm(race_ids, desc="Processing races"):
        race = df[df['raceId'] == race_id].copy()
        for _, row in race.iterrows():
            h = row['horseId']
            if h not in ratings:
                if 'handicapRatingKg' in row and not pd.isnull(row['handicapRatingKg']):
                    ratings[h] = 30.0 + (row['handicapRatingKg'] - 30.0)
                else:
                    ratings[h] = 30.0
        horses = {
            row['horseId']: {
                'rating': ratings[row['horseId']],
                'lengths_back': row['cumulative_lengths_back'],
                'weight': row['weightKg']
            }
            for _, row in race.iterrows()
        }
        surprises = {h: 0.0 for h in horses}
        n_duels   = {h: 0   for h in horses}
        horse_ids = list(horses.keys())
        for i in range(len(horse_ids)):
            for j in range(i + 1, len(horse_ids)):
                h1, h2 = horse_ids[i], horse_ids[j]
                d1, d2 = horses[h1], horses[h2]
                perf_diff = (d1['rating'] - 0.625 * d1['weight']) - (d2['rating'] - 0.625 * d2['weight'])
                bucket_exp = signed_bucket(perf_diff)
                going    = race.iloc[0].get('going_category', 'FAST') or 'FAST'
                distance = race.iloc[0].get('distance_group')
                multiplier = kg_per_length.get(going.upper(), kg_per_length['FAST']).get(distance, 1.0)
                length_gap = d2['lengths_back'] - d1['lengths_back']
                actual_diff = length_gap * multiplier
                bucket_act = signed_bucket(actual_diff)
                surprise = bucket_act - bucket_exp
                surprises[h1] += surprise
                surprises[h2] -= surprise
                n_duels[h1] += 1
                n_duels[h2] += 1
        for h in horses:
            avg_surprise = surprises[h] / max(1, n_duels[h])
            horse_row = race[race['horseId'] == h].iloc[0]
            prize_quantile = horse_row.get('prize_quantile', 0.5)
            horse_run      = horse_row.get('horse_run', 1)
            prize_bonus    = (prize_quantile - 0.5) * 5 * np.exp(-alpha * horse_run)
            ratings[h] += k_factor * (avg_surprise + prize_bonus)
            rating_history.append({'raceId': race_id, 'horseId': h, 'rating_after_race': ratings[h]})
    history_df = pd.DataFrame(rating_history)
    return df.merge(history_df, on=['raceId', 'horseId'], how='left')

## 7. Transformations & New Columns

In [11]:
# ── WebTips scoring ───────────────────────────────────────────────────────────
webTips_scored = process_webtips_simple(webTips, runners)
runners = runners.merge(
    webTips_scored, left_on=['raceId', 'horseName'],
    right_on=['race_id', 'runner'], how='left'
)
runners['runners'] = runners.groupby('raceId')['ranking'].transform('max')
runners = runners.sort_values(by='date', ascending=True)
runners['horse_run'] = runners.groupby('horseId').cumcount() + 1
runners['webTipRank_perc'] = (
    (runners['runners'] - runners['webTipRank']) / (runners['runners'] - 1)
).fillna(0)

# ── Lengths back ──────────────────────────────────────────────────────────────
margin_lookup = build_margin_lookup(runners)
runners['lengths_back'] = runners.apply(
    lambda row: margin_to_length_fast(row['margin'], row['ranking'], row['raceId'], margin_lookup),
    axis=1
)
runners['cumulative_lengths_back'] = (
    runners[runners['lengths_back'] >= 0]
    .sort_values(by='ranking')
    .groupby('raceId')['lengths_back']
    .cumsum()
)

# ── Going / distance categories ───────────────────────────────────────────────
going_mapping = {
    "Lourd": "VERY SLOW", "Très lourd": "VERY SLOW", "Collant": "VERY SLOW",
    "Souple": "SLOW",     "Très souple": "SLOW",
    "Bon souple": "FAST", "Bon": "FAST",
    "Léger": "VERY FAST", "Bon léger": "VERY FAST", "Très léger": "VERY FAST",
    "PSF Standard": "PSF", "PSF Lente": "PSF", "PSF Rapide": "PSF",
    None: None
}

for df in [runners, runners_tdy, races_tdy]:
    col = 'going' if 'going' in df.columns else None
    if col:
        df['going_category'] = df[col].map(going_mapping)
        df['distance_group'] = df['distance'].apply(get_distance_group)

runners = runners[runners['ranking'].isnull()==False]

# ── Additional runner columns ─────────────────────────────────────────────────
runners['cumulative_lengths_back'] = runners['cumulative_lengths_back'].fillna(0)
runners['pos_perc']   = (runners['runners'] - runners['ranking']) / (runners['runners'] - 1)
runners['place']      = np.where(runners['ranking'] <= 2, 1,
                          np.where((runners['ranking'] == 3) & (runners['runners'] >= 7), 1, 0))
runners['win']        = np.where(runners['ranking'] <= 1, 1, 0)
runners['prizemoney'] = runners.apply(
    lambda row: calculate_prize(row['totalPrize_y'], row['ranking']), axis=1
)
runners['handicapRating_adj'] = runners['handicapRatingKg'] + 0.625 * (55 - runners['weightKg'])
runners['date']       = pd.to_datetime(runners['date'])
runners               = runners.sort_values(by='date')
runners['cumulative_lengths_back'] = pd.to_numeric(runners['cumulative_lengths_back'], errors='coerce')
runners['odds']       = 1 / runners['liveOdd']
runners['odds_sum']   = runners.groupby(['raceId', 'name_meeting'])['odds'].transform('sum')
runners['place_sp']   = (runners['liveOdd'] - 1) / 4 + 1
runners['odds_place'] = 1 / runners['place_sp']

if 'name_meeting' in runners.columns and 'meetingName' not in runners.columns:
    runners = runners.rename(columns={'name_meeting': 'meetingName'})
runners['runners'] = runners.groupby(['raceId', 'date'])['horseName'].transform('count')
runners['pos_perc']  = (runners['runners'] - runners['ranking']) / (runners['runners'] - 1)

# ── draw_unique ───────────────────────────────────────────────────────────────
def make_draw_unique(row, meeting_col):
    parts = [row['draw'], row[meeting_col], row['distance'],
             row.get('direction'), row.get('surface')]
    return ' - '.join(str(x) for x in parts if pd.notnull(x))

runners['draw_unique'] = runners.apply(make_draw_unique, axis=1, meeting_col='name_meeting')

merged_temp = runners_tdy.merge(
    races_tdy[['id_race', 'direction', 'rail', 'surface']],
    left_on='raceId', right_on='id_race', how='left'
)
runners_tdy['draw_unique'] = merged_temp.apply(
    make_draw_unique, axis=1, meeting_col='meetingName'
)

Processing tips:   0%|          | 0/24152 [00:00<?, ?it/s]

## 8. Build Ratings

In [12]:
# Load cached state
df_with_ratings = pd.read_parquet(f"{BASE}/df_with_ratings.parquet")

with open(f"{BASE}/ratings_state.json") as f:
    saved_ratings = {int(k): v for k, v in json.load(f).items()}

with open(f"{BASE}/ratings_last_date.txt") as f:
    last_processed_date = f.read().strip()

print(f"📂 Loaded {len(df_with_ratings)} cached rows (up to {last_processed_date})")

# Find new races not yet in the cache
new_runners = runners[
    (runners['date'] >= '2021-01-01') &
    (runners['date'] > last_processed_date) &
    (~runners['raceId'].isin(df_with_ratings['raceId']))
]

if new_runners.empty:
    print("✅ No new races — using cached ratings.")
else:
    print(f"⚙️  Processing {new_runners['raceId'].nunique()} new races ({len(new_runners)} rows)...")

    new_rated = update_ratings_with_history(
        new_runners, k_factor=0.5, initial_ratings=saved_ratings
    )
    new_rated['rating_after_race'] = new_rated['rating_after_race'].round(1)
    new_rated['draw_unique'] = new_rated.apply(
        make_draw_unique, axis=1, meeting_col='name_meeting'
    )

    # Append and save
    df_with_ratings = pd.concat([df_with_ratings, new_rated], ignore_index=True)
    df_with_ratings.to_parquet(f"{BASE}/df_with_ratings.parquet", index=False)

    # Update ratings state
    updated_ratings = (
        new_rated.sort_values('date')
        .groupby('horseId')['rating_after_race']
        .last()
        .to_dict()
    )
    saved_ratings.update({int(k): v for k, v in updated_ratings.items()})
    with open(f"{BASE}/ratings_state.json", "w") as f:
        json.dump({str(k): v for k, v in saved_ratings.items()}, f)

    with open(f"{BASE}/ratings_last_date.txt", "w") as f:
        f.write(str(df_with_ratings['date'].max()))

    print(f"✅ Appended {len(new_rated)} new rows. Total: {len(df_with_ratings)}")

📂 Loaded 259867 cached rows (up to 2026-04-20 00:00:00)
⚙️  Processing 8 new races (104 rows)...


Processing races:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Appended 104 new rows. Total: 259971


### 8.1 Backfill handicapRatingKg

In [13]:
import pandas as pd
import numpy as np
import gc

TOL = 0.075

runners_sorted = runners.sort_values('date')

first_non_null = (
    runners_sorted[runners_sorted['handicapRatingKg'].notna()]
    .groupby('horseId')
    .first()[['handicapRatingKg']]
    .reset_index()
)

bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.001]
labels = ['0.0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0']
runners['pos_perc_bin'] = pd.cut(runners['pos_perc'], bins=bins, labels=labels, right=False)

SENTINEL = '__NULL__'
for col, sentinel in [('class', SENTINEL), ('type', SENTINEL), ('age', -1)]:
    runners[f'{col}_join'] = runners[col].fillna(sentinel)

null_rows = runners[runners['handicapRatingKg'].isna()].copy()
null_rows['_orig_index'] = null_rows.index

null_pool = runners[runners['handicapRatingKg'].isna()][
    ['horseId', 'pos_perc_bin', 'age_join', 'class_join', 'type_join', 'totalPrize_y']
].copy()
null_pool_with_rating = null_pool.merge(first_non_null, on='horseId', how='inner')

null_rows_keyed = null_rows[
    ['horseId', 'pos_perc_bin', 'age_join', 'class_join', 'type_join', 'totalPrize_y', '_orig_index']
].copy()

JOIN_KEYS = ['pos_perc_bin', 'age_join', 'class_join', 'type_join']
CHUNK_SIZE = 50_000
results = []

for start in range(0, len(null_rows_keyed), CHUNK_SIZE):
    chunk = null_rows_keyed.iloc[start : start + CHUNK_SIZE]

    merged = chunk.merge(
        null_pool_with_rating[JOIN_KEYS + ['totalPrize_y', 'horseId', 'handicapRatingKg']],
        on=JOIN_KEYS,
        suffixes=('_runner', '_ref')
    )

    merged = merged[merged['horseId_runner'] != merged['horseId_ref']]

    prize_ok = merged['totalPrize_y_ref'].between(
        merged['totalPrize_y_runner'] * (1 - TOL),
        merged['totalPrize_y_runner'] * (1 + TOL)
    )
    merged = merged[prize_ok]

    if not merged.empty:
        agg_chunk = (
            merged
            .groupby('_orig_index')['handicapRatingKg']
            .agg(fill_value='mean', fill_n='count')
        )
        results.append(agg_chunk)

    del merged
    gc.collect()

if results:
    combined = pd.concat(results)
    agg = combined.groupby(level=0).agg(
        fill_value=('fill_value', 'mean'),
        fill_n=('fill_n', 'sum')
    )

runners['handicapRatingKg_filled'] = runners['handicapRatingKg'].fillna(agg['fill_value'])
runners['handicapRatingKg_fill_n'] = agg['fill_n'].reindex(runners.index).astype('Int64')
runners['handicapRatingKg'] = runners['handicapRatingKg'].fillna(runners['handicapRatingKg_filled'])

runners = runners.drop(
    columns=['pos_perc_bin', 'class_join', 'age_join', 'type_join',
             'handicapRatingKg_filled', 'handicapRatingKg_fill_n'],
    errors='ignore'
)

### 8.2 Calculate ARR

In [14]:
from math import log
from tqdm import tqdm

def calculate_arr(runners: pd.DataFrame) -> pd.DataFrame:
    runners = runners.copy()
    runners['ARR'] = np.nan

    groups = list(runners.groupby('raceId'))

    for race_id, race in tqdm(groups, desc='Calculating ARR'):
        going   = race['going_category'].iloc[0]
        d_group = race['distance_group'].iloc[0]

        if pd.isna(going) or pd.isna(d_group):
            continue
        if going not in kg_per_length or d_group not in kg_per_length[going]:
            continue

        kpl  = kg_per_length[going][d_group]
        mask = (race['pos_perc'].values > 0.66)

        if not mask.any():
            continue

        clb  = np.maximum(race['cumulative_lengths_back'].values, 0)
        wkg  = race['weightKg'].values
        hcap = race['handicapRatingKg'].values

        length_diff = clb[mask][:, None] - clb[None, :]

        with np.errstate(divide='ignore', invalid='ignore'):
            log_factor = np.where(
                length_diff > 0,
                np.maximum(np.log(length_diff) / np.log(4), 1.0),
                1.0
            )

        weight_diff = wkg[None, :] - wkg[mask][:, None]
        hcap_r = hcap[mask][:, None]
        ref_matrix = length_diff * kpl / log_factor + weight_diff * kpl + hcap_r

        ref_indices = np.where(mask)[0]
        all_indices = np.arange(len(race))
        diag_mask   = (ref_indices[:, None] == all_indices[None, :])
        ref_matrix  = np.where(diag_mask, hcap_r, ref_matrix)

        arr_vals = np.nanmean(ref_matrix, axis=0)
        arr_vals = np.where(np.isnan(arr_vals), np.nan, np.maximum(arr_vals, 0))

        runners.loc[race.index, 'ARR'] = np.round(arr_vals,2)

    return runners

runners = calculate_arr(runners)

Calculating ARR:   5%|▌         | 1531/28740 [00:01<00:31, 868.36it/s]/tmp/ipykernel_832/4008664116.py:47: RuntimeWarning: Mean of empty slice
  arr_vals = np.nanmean(ref_matrix, axis=0)
Calculating ARR:  12%|█▏        | 3465/28740 [00:03<00:21, 1190.83it/s]/tmp/ipykernel_832/4008664116.py:47: RuntimeWarning: Mean of empty slice
  arr_vals = np.nanmean(ref_matrix, axis=0)
Calculating ARR:  82%|████████▏ | 23431/28740 [00:22<00:10, 513.63it/s]/tmp/ipykernel_832/4008664116.py:47: RuntimeWarning: Mean of empty slice
  arr_vals = np.nanmean(ref_matrix, axis=0)
Calculating ARR:  96%|█████████▋| 27734/28740 [00:26<00:00, 1304.64it/s]/tmp/ipykernel_832/4008664116.py:47: RuntimeWarning: Mean of empty slice
  arr_vals = np.nanmean(ref_matrix, axis=0)
Calculating ARR:  98%|█████████▊| 28139/28740 [00:26<00:00, 1286.75it/s]/tmp/ipykernel_832/4008664116.py:47: RuntimeWarning: Mean of empty slice
  arr_vals = np.nanmean(ref_matrix, axis=0)
Calculating ARR:  98%|█████████▊| 28269/28740 [00:26<00:00,

## 9. Save Processed runners

In [15]:
# Save the fully processed historical runners — loaded by the fast HTML notebook
runners.to_parquet(f"{BASE}/runners_processed.parquet", index=False)
print(f"✅ Saved runners_processed.parquet  ({len(runners):,} rows)")

✅ Saved runners_processed.parquet  (302,714 rows)


## 10. Load HTML Functions & Compute Notepad Flags

In [16]:
%run "/content/drive/MyDrive/PT/pt_html_functions.py"

In [ ]:
notepad_flags = compute_notepad_flags(df_today=runners_tdy, runners_hist=runners)

# Save for fast notebook
import pickle
from datetime import date
TODAY = date.today().strftime("%Y-%m-%d")
with open(f"{BASE}/notepad_flags_{TODAY}.pkl", "wb") as f:
    pickle.dump(notepad_flags, f)
print(f"✅ Saved notepad_flags_{TODAY}.pkl  ({len(notepad_flags)} flags)")

🤖 compute_notepad_flags: 233 races → 12 API call(s)
  batch 1/12: 35 results parsed
  batch 2/12: 25 results parsed
  batch 3/12: 26 results parsed
  batch 4/12: 24 results parsed
  batch 5/12: 24 results parsed
  batch 6/12: 29 results parsed
  batch 7/12: 30 results parsed
  batch 8/12: 24 results parsed
  batch 9/12: 21 results parsed
  batch 10/12: 24 results parsed


In [ ]:
# ── Export base HTML + collect race JSONs (single computation pass) ──────────
import os
BASE_OUT = f"{BASE}/races"
os.makedirs(BASE_OUT, exist_ok=True)

saved_files, race_jsons = export_all_races_html(
    df_hist          = runners,
    df_today         = runners_tdy,
    webTips_tdy      = webTips_tdy,
    today_tips       = today_tips,
    races_tdy        = races_tdy,
    df_with_ratings  = df_with_ratings,
    notepad_flags    = notepad_flags,
    pmu_odds_history = None,
    output_dir       = BASE_OUT,
)
print(f'\n✅ {len(saved_files)} HTML files exported with verdict placeholders')
print(f'   Race JSONs available: {len(race_jsons)} races')


In [ ]:
# ── Generate AI verdicts per race (race JSONs from HTML export, no recomputation) ─
import time

all_verdicts = {}  # {horse_name: verdict_text}
_total = len(race_jsons)
_idx   = 0

for race_key, race_json in race_jsons.items():
    _idx += 1
    print(f'[{_idx}/{_total}] {race_json.get("meeting","")} — {race_json.get("race","")}',
          end=' ... ', flush=True)
    try:
        _vd = generate_race_verdicts(race_json, api_key=ANTHROPIC_API_KEY)
        all_verdicts.update(_vd)
        print(f'✅ {len(_vd)} horses')
    except Exception as _e:
        print(f'❌ {_e}')
    time.sleep(0.5)  # gentle rate limiting

print(f'\n✅ Verdicts: {len(all_verdicts)} horses across {_total} races')


In [ ]:
# ── Inject verdicts into HTML + save precomputed data ────────────────────────
update_verdicts_in_html(BASE_OUT, TODAY, all_verdicts)

import pickle
precomputed_tdy = {
    'runners_tdy': runners_tdy,
    'webTips_tdy' : webTips_tdy,
    'odds_tdy'    : odds_tdy,
    'races_tdy'   : races_tdy,
    'today_tips'  : today_tips,
    'all_verdicts': all_verdicts,
}
with open(f"{BASE}/precomputed_tdy_{TODAY}.pkl", "wb") as f:
    pickle.dump(precomputed_tdy, f)
print(f"\n✅ Saved precomputed_tdy_{TODAY}.pkl  (incl. {len(all_verdicts)} verdicts)")
print(f"✅ Base HTMLs (with AI verdicts) saved to: {BASE_OUT}")


In [ ]:
print("\n✅ Vorarbeiten complete. Files written to BASE:")
print(f"   runners_processed.parquet")
print(f"   notepad_flags_{TODAY}.pkl")
print(f"   precomputed_tdy_{TODAY}.pkl")
print(f"   races/*.html  (base HTMLs, PMU placeholders)")
print("\nNow run PT_Create_HTMLs_fast.ipynb ▶")
